## Izvlacenje dataset-a

In [303]:
import pandas as pd

In [370]:
df = pd.read_excel("../telekom_korisnici.xlsx",sheet_name=["Podaci","Rečnik_podataka"])
data = df["Podaci"].copy()

data_dict = df["Rečnik_podataka"].copy()

display(data.describe())

,godine,staz_meseci,minuti_poziva,broj_sms,broj_reklamacija,ocena_zadovoljstva
count,1184.000000,1232.000000,1232.000000,1232.000000,1232.000000,1136.000000
mean,42.237331,41.548701,279.222403,38.410714,0.758929,3.617958
std,14.925691,39.504476,431.454944,6.144891,1.033713,0.981859
min,-26.000000,-9.000000,-139.000000,20.000000,0.000000,1.000000
25%,32.000000,14.000000,162.000000,34.000000,0.000000,3.000000
50%,42.500000,29.000000,258.500000,38.000000,0.000000,4.000000
75%,52.000000,56.250000,345.250000,43.000000,1.000000,4.000000
max,150.000000,180.000000,7500.000000,59.000000,7.000000,5.000000


# Standardizacija

In [371]:
categories=["region","pol", "tip_paketa", "kanal_prodaje", "churn", "id_korisnika"]

for cat in categories:
    data[cat]=data[cat].astype("string").str.strip()

# Fala fajlu `problemi.csv` :)
map_region = {
    "beograd":"Beograd",
    "bg":"Beograd",
    "vojvodina":"Vojvodina",
    "šumadija i zapadna srbija":"Šumadija i Zapadna Srbija",
    "sumadija i zapadna srbija":"Šumadija i Zapadna Srbija",
    "južna i istočna srbija":"Južna i Istočna Srbija",
    "juzna i istocna srbija":"Južna i Istočna Srbija"
}

data["region"] = data["region"].str.lower().map(map_region)

map_pol={
    "m":"M",
    "ž":"Ž",
    "z":"M",
    "muski":"M",
    "muški":"M",
    "zenski":"Ž",
    "ženski":"Ž"
}

data["pol"]=data["pol"].str.lower().map(map_pol)

map_chrun={
    "yes":"Da",
    "1":"Da",
    "no":"Ne",
    "0":"Ne",
    "ne":"Ne",
    "da":"Da"
}
data["churn"] = data["churn"].str.lower().map(map_chrun)

map_tip={
    'post-paid':'Postpaid',
    'postpaid':'Postpaid',
    'pre-paid':'Prepaid',
    'biznis':'Biznis',
    'business':'Biznis',
    'prepaid':'Prepaid'
}

data["tip_paketa"]= data["tip_paketa"].str.lower().map(map_tip)

map_kanal={
    'poslovnica':'Poslovnica',
    'online':'Online',
    'call centar':'Call centar',
    'partner':'Partner'
}
data["kanal_prodaje"]=data["kanal_prodaje"].str.lower().map(map_kanal)

allowed = dict(zip(data_dict["kolona"], data_dict["dozvoljene vrednosti"].str.split("; ")))

categories=["region","pol", "tip_paketa", "kanal_prodaje", "churn"]

for col in categories:
    assert data[col].dropna().isin(allowed[col]).all(), col
    print(f"{col} {sorted(data[col].dropna().unique())}")



region ['Beograd', 'Južna i Istočna Srbija', 'Vojvodina', 'Šumadija i Zapadna Srbija']
pol ['M', 'Ž']
tip_paketa ['Biznis', 'Postpaid', 'Prepaid']
kanal_prodaje ['Call centar', 'Online', 'Partner', 'Poslovnica']
churn ['Da', 'Ne']


## Uklanjanje duplikata

In [372]:
pre = len(data)

data = data.drop_duplicates()

print("uklonjeno potpunih {}".format(pre-len(data)))
print("preostalih dupliranih id: {}".format(data["id_korisnika"].duplicated().sum()))

uklonjeno potpunih 32
preostalih dupliranih id: 0


## Tekst u brojeve

In [373]:
#priznajem da sam se pomogao uputstvom
import numpy as np

def u_broj(v):
    if pd.isna(v):
        return np.nan
    if isinstance(v,(float,int)):
        return float(v)

    num = str(v).lower().replace("rsd","").replace("din","").strip()

    if "," in num and "." in num:
        num=num.replace(".","").replace(",",".")

    elif "," in num:
        num=num.replace(",",".")

    pos = num.rfind(".")
    size = len(num)

    num = num.replace(".", "")
    num = int(num) / np.power(10, size - pos - 1)

    return float(num)



for col in ["mesecni_racun_rsd", "potrosnja_gb"]:
    nan_pre = data[col].isna().sum()
    data[col]= data[col].map(u_broj)
    print(f"{col} {nan_pre} {data[col].isna().sum()}")


mesecni_racun_rsd 36 36
potrosnja_gb 36 36


## Parsiranje datuma

In [374]:
datumi = data["datum_aktivacije"].astype("string").str.strip()

iso_std = pd.to_datetime(datumi,format="%Y-%m-%d",errors="coerce")

serbian = pd.to_datetime(datumi,format="%d.%m.%Y.",errors="coerce")

us = pd.to_datetime(datumi,format="%m/%d/%Y",errors="coerce")

not_valid = iso_std > pd.Timestamp("2026-09-01")

datumi = iso_std.fillna(serbian).fillna(us) #popuni falinke u iso parsiranju sa serbianom ili us-om

print(f"ISO {iso_std.notna().sum()} SRB {serbian.notna().sum()} US {us.notna().sum()}")

print(f"Nevalidni {not_valid.isna().sum()}") #logicno da nema, jer smo zakrpili necim pre 2 reda


ISO 1030 SRB 110 US 60
Nevalidni 0


## Nelogične vrednosti

In [375]:
# Sporne kolone
# godine, staz_meseci, mesecni_racun_rsd minuti_poziva

failed_cols={
    "godine": lambda s:(s<18) | (s>100),
    "staz_meseci": lambda s: s<1,
    "mesecni_racun_rsd": lambda s: s<0,
    "minuti_poziva": lambda s: (s<0) | (s>5000),
    "potrosnja_gb": lambda s: (s<0) |  (s>150),
    "broj_sms": lambda s: (s<0),
}

for col,rule in failed_cols.items():
    result = rule(data[col]) & data[col].notna()
    print(col,result.sum(), data.loc[result,col].head().tolist())
    data.loc[result,col] = np.nan

ten = data['mesecni_racun_rsd']>15000
print(len(data.loc[ten,'mesecni_racun_rsd']))


godine 7 [-26.0, 127.0, 135.0, -19.0, 150.0]
staz_meseci 5 [-4, -9, -9, -3, -3]
mesecni_racun_rsd 0 []
minuti_poziva 8 [-139, -98, 7500, 7500, -62]
potrosnja_gb 5 [850.0, 999.9, 920.5, 999.9, 920.5]
broj_sms 0 []
9


Kako je $\frac{9}{1260} < 1 \%$ u sustini to je zanemarivo malo s aspekta obima uzorka.

In [376]:
data_copy = data.copy()
print(stat,p_value)
display(data_copy["mesecni_racun_rsd"].describe())

records = data_copy.loc[ten]

data_copy = data_copy.drop(records.index)

display(data_copy["mesecni_racun_rsd"].describe())


print(stat,p_value)

0.9183899941053502 1.769233464734458e-24


count     1164.000000
mean      2961.697167
std       3880.865830
min          0.125000
25%       1366.000000
50%       2359.000000
75%       3530.000000
max      72090.000000
Name: mesecni_racun_rsd, dtype: float64

count    1155.000000
mean     2683.701733
std      1725.109720
min         0.125000
25%      1359.500000
50%      2342.000000
75%      3511.500000
max      9675.000000
Name: mesecni_racun_rsd, dtype: float64

0.9183899941053502 1.769233464734458e-24


Naknadnom analizom je utvrdjeno da se normalnost raspodele velicine `mesecni_racun_rsd` se ne menja, jer je Shapiro-Wilco prijavio za vrednosti (pre i posle):
```
0.3811855574572757 1.4706799491645429e-52
0.9183899941053502 1.769233464734458e-24
```
sto za nivo znacajnosti od $5\%$ govori da raspodela NIJE normalna prema tome prosek ne moze biti dobar pokazatelj. Odluka o izbacivanju zavisi od toga koliko su druge velicine pogodjene ovom promenom (da li sadrze outlier-e i sl). 

In [301]:
display(data.loc[ten])

,id_korisnika,region,pol,godine,tip_paketa,staz_meseci,mesecni_racun_rsd,potrosnja_gb,minuti_poziva,broj_sms,broj_reklamacija,kanal_prodaje,datum_aktivacije,ocena_zadovoljstva,churn
108,K00318,Vojvodina,M,47.0,Postpaid,15.0,40630.0,9.7,430.0,36.0,2,Online,2025-05-19,3.0,Ne
131,K00388,Južna i Istočna Srbija,M,23.0,Postpaid,38.0,32910.0,8.6,383.0,36.0,0,Online,2023-07-03,5.0,Ne
368,K00630,Beograd,M,33.0,Postpaid,24.0,31070.0,12.4,207.0,41.0,0,Online,2024-08-27,4.0,Ne
413,K00052,Vojvodina,M,35.0,Postpaid,35.0,29740.0,12.1,337.0,32.0,0,Online,2023-09-10,4.0,Ne
587,K00368,Beograd,M,24.0,Biznis,12.0,64490.0,6.3,350.0,48.0,0,Poslovnica,2025-08-31,4.0,Da
598,K00361,Šumadija i Zapadna Srbija,M,23.0,Postpaid,27.0,20750.0,7.9,253.0,35.0,0,Online,2024-05-09,5.0,Ne
689,K00197,Južna i Istočna Srbija,M,55.0,Postpaid,86.0,32020.0,8.4,223.0,37.0,0,Poslovnica,2019-07-06,3.0,Ne
802,K00632,Vojvodina,Ž,37.0,Biznis,10.0,72090.0,5.1,343.0,50.0,0,Poslovnica,2025-10-21,5.0,Ne
945,K00011,Južna i Istočna Srbija,Ž,47.0,Postpaid,65.0,24040.0,10.8,342.0,43.0,0,Partner,2021-03-24,5.0,Ne
